WE will build a sample neural network 

We will learn how to build a simple neural network pipeline


In [ ]:
import pandas as pd
import torch 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()


In [ ]:
'''
Code flow

1. load the dataset
2. basic programming
3. training process
   a. create the model
   b. forward pass
   c. back propagation
   d. parameters update
4. Model Evaluation


'''

In [ ]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)



Train Test Split

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

Scaling

In [ ]:
scaler= StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

Label encoding

In [ ]:
encoder= LabelEncoder()
y_train= encoder.fit_transform(y_train)
y_test= encoder.transform(y_test)

In [ ]:
y_test

Numpy arrays to PyTorch tensors

In [ ]:
X_train_tensor= torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor= torch.from_numpy(y_train)
y_test_tensor= torch.from_numpy(y_test)

In [ ]:
X_train_tensor

Define the Model

In [ ]:
class MySimpleNN():
    def __init__(self, X):
        self.weights= torch.rand(X.shape[1],1,dtype= torch.float64, requires_grad= True)

        self.bias= torch.zeros(1,dtype= torch.float64, requires_grad= True)
    
    def forward(self,X):
        z= torch.matmul(X,self.weights )+ self.bias
        y_pred= torch.sigmoid(z)

        return y_pred
    
    def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
       epsilon = 1e-7
       y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
       loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
       return loss

Important parameters

In [ ]:
learning_rate= 0.1
epochs=25

Training Pipeline

In [ ]:
#Create Model

model= MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):
  #forward pass
  y_pred= model.forward(X_train_tensor)

  # loss calucation
  loss= model.loss_function(y_pred,y_train_tensor)
  
  # backpropagation
  
  loss.backward()

  #parameters update
  with torch.no_grad():
    model.weights-=learning_rate*model.weights.grad
    model.bias-=learning_rate*model.bias.grad
  
  #zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  #print loss in each epoch
  print(f"epoch: {epoch+1}, Loss: {loss.item()}")


In [ ]:
model.weights

Evaluation


In [ ]:
#Model Evaluation
with torch.no_grad():
    y_pred= model.forward(X_test_tensor)
    y_pred= (y_pred>0.5).float()
    accuracy= (y_pred== y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy.item()}")
print(y_pred)